# 1b - Company A, built by hand

**Time:** 25 minutes build + 10 minutes check  |  **AI: OFF** (no Gemini, no ChatGPT in this notebook)

**What you will do**
- Build four small functions, one per step: `dividends`, `terminal_value`, `discount`, `assessment`.
- Chain them into a value for company A, and compare it to the market price.
- Break it on purpose (g = r), add a guard, then find out who is right: 7.69 or 8.54?

Each blank is marked `# YOUR CODE`. Replace the `...` (it means "nothing here yet") with your code. Run each function's cell (Shift+Enter), then the test cell below it, and compare with the number in the text. If a test gives an error or a wrong number, look at the function above it first.

## The problem

Company A paid a dividend of **0.27** last year. Dividends grow **10% a year for 3 years**, then **8% a year forever**. The required return is **12%**. The share price is **9.74**.

What is a share worth? Is the market price too high?

| Step | Function | You should get |
|---|---|---|
| 1 | `dividends(d0, g, years)` | 0.297, 0.327, 0.359 |
| 2 | `terminal_value(last_div, g, r)` | 9.703 |
| 3 | `discount(cash_flows, r)` | value today **7.69** |
| 4 | `assessment(value, price)` | "overvalued" |

In [ ]:
# The assumptions tab. Rates are decimals.
inputs = {
    "d0": 0.27,       # last dividend paid
    "g_high": 0.10,   # growth in years 1 to 3
    "years": 3,       # number of high-growth years
    "g_long": 0.08,   # growth forever, after year 3
    "r": 0.12,        # required return
    "price": 9.74,    # market price today
}

## Step 1 - Dividends in the high-growth years

The dividend of year `t` is `d0 * (1 + g) ** t`, for `t` = 1, 2, 3. Build a list with one dividend per year.

Hints:
- `range(1, years + 1)` gives 1, 2, ..., `years`. Start at 1, because the first new dividend is year 1 (not year 0).
- `divs = []` is an empty list. Add each dividend at the end with `divs.append(...)`.
- Your loop goes inside the function, so indent it one more step, like the other lines.

You should get **0.297, 0.327, 0.359**.

In [ ]:
def dividends(d0, g, years):
    divs = []
    # YOUR CODE: for every year t from 1 to years, add d0 * (1 + g) ** t to the list
    ...
    return divs

In [ ]:
divs = dividends(inputs["d0"], inputs["g_high"], inputs["years"])
for d in divs:
    print(f"{d:.3f}")

## Step 2 - Terminal value at the end of year 3

After year 3, dividends grow at `g` forever. The value of that stream, at the end of year 3, is (Gordon growth):

`terminal value = last dividend * (1 + g) / (r - g)`

Careful: it uses **next year's** dividend, so `last_div * (1 + g)`, not `last_div`.

You should get **9.703**.

In [ ]:
def terminal_value(last_div, g, r):
    return ...   # YOUR CODE: next year's dividend divided by (r - g)

In [ ]:
tv = terminal_value(divs[-1], inputs["g_long"], inputs["r"])
print(f"Terminal value at year 3: {tv:.3f}")

## Step 3 - Discount to today

`discount(cash_flows, r)` adds up the present value of a list of cash flows. The first item is year 1, the second year 2, and so on. Use `enumerate(cash_flows, start=1)`.

It works like the NPV loop in 1a (cell 6), with `total` as the running total.

The terminal value sits at year 3, so it is discounted 3 years, like the last dividend.

You should get **7.69**.

In [ ]:
def discount(cash_flows, r):
    total = 0
    # YOUR CODE: for each cash flow, add cf / (1 + r) ** t to total (t starts at 1)
    ...
    return total

In [ ]:
# value today = dividends of years 1 to 3 + terminal value (also 3 years away)
value = discount(divs, inputs["r"]) + tv / (1 + inputs["r"]) ** inputs["years"]
print(f"Value of one share: {value:.2f}")

## Step 4 - Is the share overvalued?

`assessment(value, price)` compares the two and answers with one word:
- **"overvalued"** if the price is more than 5% above the value (you would pay too much),
- **"undervalued"** if the price is more than 5% below the value,
- **"fair"** otherwise.

Use `if / elif / else`. `elif` means "else, if": a second test, tried only when the first one is false. Words are text, so they go in quotes: `return "overvalued"`.

Company A: value 7.69, price 9.74, so you should get **"overvalued"**.

In [ ]:
def assessment(value, price):
    # YOUR CODE: if price is above value * 1.05 -> "overvalued"
    #            elif price is below value * 0.95 -> "undervalued"
    #            else -> "fair"
    ...

In [ ]:
print(f"Value {value:.2f} vs price {inputs['price']}: {assessment(value, inputs['price'])}")

## The extreme case: what if g = r?

Gordon growth needs the required return to be **above** the growth rate. If `g = r`, the bottom of the formula is `r - g = 0`.

**This cell fails on purpose.** Read the message: last line first, then the arrow.

**Your prediction:** which error will Python raise?

In [ ]:
terminal_value(0.359, 0.12, 0.12)

`ZeroDivisionError: float division by zero`: the formula divided by 0. This is the second error type to know (the first was `TypeError`).

A model should not crash with a cryptic message, and it should never give a silly number silently. It should **stop and say why**, in finance words. Add a **guard** at the top of the function: if `g >= r`, stop with your own message: `raise ValueError("...")`. `raise` stops the code and shows the message you wrote.

(Checks card, entry 3: extreme case, g = r must fail loudly.)

In [ ]:
def terminal_value(last_div, g, r):
    if ...:   # YOUR CODE: the condition g >= r
        raise ValueError("Growth must be below the required return (g < r).")
    return ...   # YOUR CODE: the same formula as in step 2

In [ ]:
# Test 1: the normal case still works
print(f"{terminal_value(divs[-1], inputs['g_long'], inputs['r']):.3f}")

**Test 2 fails on purpose**, but now with **your** message, in finance words. Compare it with the `ZeroDivisionError` above: which one would you rather see at 2am?

In [ ]:
terminal_value(0.359, 0.12, 0.12)

## Check: 7.69 or 8.54?

Another notebook in this course values company A at **8.54**, with the same inputs. Yours says 7.69. Who is right?

That notebook reads its inputs from an Excel sheet. Here is a small dictionary standing in for that sheet, and the code from that notebook, adapted to your functions:

| Cell | Label in the sheet | Value |
|---|---|---|
| B7 | Perpetual growth | 0.12 |
| B8 | WACC | 0.08 |

In [ ]:
sheet = {"B7": 0.12, "B8": 0.08}   # stands in for the Excel sheet

growth = sheet["B7"]   # label in the sheet: "Perpetual growth"
wacc = sheet["B8"]     # label in the sheet: "WACC"

# The code from that notebook, adapted to your functions
tv_other = terminal_value(divs[-1], wacc, growth)
value_other = discount(divs, wacc) + tv_other / (1 + wacc) ** 3

print(f"Terminal value: {tv_other:.3f}")
print(f"Value: {value_other:.2f}")

The terminal value is 9.703, the same as yours. But the value is 8.54, not 7.69.

**Your answer:** there are two errors. What is each one? Why does the terminal value look right? Where does the gap of 0.85 come from? Write 3 to 4 lines here: ...

Now read the sheet the right way round, and run the numbers again.

You should get **7.69**.

In [ ]:
growth_right = ...     # YOUR CODE: which cell holds the growth?
required_right = ...   # YOUR CODE: which cell holds the required return?

tv_right = terminal_value(divs[-1], growth_right, required_right)
value_right = discount(divs, required_right) + tv_right / (1 + required_right) ** 3
print(f"Value: {value_right:.2f}")

## What you learned

**Python**
- **Functions**, one per step (`dividends`, `terminal_value`, `discount`, `assessment`), chained into a value.
- **`range(1, n + 1)`** and **`enumerate(..., start=1)`**: counting from year 1.
- **Condition** with three outcomes (`if / elif / else`), and a **guard** (`if g >= r: raise ValueError(...)`) that stops the code with your message.
- Three **error types**: `TypeError` (wrong kind of value), `ZeroDivisionError` (divide by 0), `ValueError` (your guard).

**Finance**
- Gordon growth uses next year's dividend: `last * (1 + g) / (r - g)`. It needs r above g.
- Company A is worth 7.69 and trades at 9.74: overvalued.
- Two errors can hide each other: swapped labels plus swapped arguments gave a plausible 8.54. Always check a number a second way.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right